# NHL Player Dimension Explorer

**Issue #162** — Data exploration artifact for the NHL Dashboard analytics pipeline.

Compares player dimension data from three NHL API endpoints to inform the `player`
dimension schema and backfill workflow before any pipeline code is written.

## Overview

Three endpoints are compared:

| Endpoint | Role |
|---|---|
| `/v1/roster-season/{team}` | Discover which seasons have roster data |
| `/v1/roster/{team}/{season}` | Bulk backfill — 32 calls covers all active NHL players |
| `/v1/player/{player_id}/landing` | Per-player enrichment: `currentTeamId`, `isActive`, draft details |
| `/v1/gamecenter/{game_id}/boxscore` | Side-effect of game ingestion — player identity quality check |

## Setup

```bash
pip install jupyter pandas httpx
jupyter notebook nhl-dashboard/notebooks/player_dim_explorer.ipynb
```

Run all cells top-to-bottom. The batch fetch in Section 6 calls the live NHL API
for all 32 teams — expect ~60–90 s total depending on network speed.

## Section 1 — Setup

Imports, constants, and sample IDs used throughout the notebook.
All 32 NHL team abbreviations are listed here for the batch fetch in Section 6.

In [ ]:
import json
import time
from pprint import pprint

import httpx
import pandas as pd

NHL_BASE = "https://api-web.nhle.com/v1"

# All 32 NHL team abbreviations for bulk backfill scan
NHL_TEAMS = [
    "ANA", "BOS", "BUF", "CGY", "CAR", "CHI", "COL", "CBJ",
    "DAL", "DET", "EDM", "FLA", "LAK", "MIN", "MTL", "NSH",
    "NJD", "NYI", "NYR", "OTT", "PHI", "PIT", "SJS", "SEA",
    "STL", "TBL", "TOR", "UTA", "VAN", "VGK", "WSH", "WPG",
]

SAMPLE_TEAM    = "TOR"
SAMPLE_SEASON  = "20252026"
SAMPLE_GAME_ID = 2025020001

# Populated in Section 3 after fetching a roster
SAMPLE_PLAYER_ID = None

print(f"Teams configured : {len(NHL_TEAMS)}")
print(f"Sample team      : {SAMPLE_TEAM}")
print(f"Sample season    : {SAMPLE_SEASON}")
print(f"Sample game      : {SAMPLE_GAME_ID}")

## Section 2 — API Exploration: `/v1/roster-season/{team}`

This supporting endpoint returns which seasons have roster data for a given team.
Use it to determine backfill scope before calling the full roster endpoint.

**Expected response:** a list of season integers, e.g. `[20212022, 20222023, ...]`.

In [ ]:
resp = httpx.get(f"{NHL_BASE}/roster-season/{SAMPLE_TEAM}", timeout=15)
resp.raise_for_status()
roster_seasons = resp.json()

print(f"Available roster seasons for {SAMPLE_TEAM}:")
if isinstance(roster_seasons, list):
    print(f"  Count          : {len(roster_seasons)}")
    print(f"  Earliest season: {min(roster_seasons)}")
    print(f"  Most recent    : {max(roster_seasons)}")
    print(f"  All seasons    : {roster_seasons}")
else:
    print(f"  Response type  : {type(roster_seasons)}")
    pprint(roster_seasons)

## Section 3 — API Exploration: `/v1/roster/{team}/{season}`

The primary source for bulk player dim population. A single call per (team, season)
returns all active roster members — forwards, defensemen, and goalies — with rich
biographical data.

**32 calls covers the entire active NHL player population.**

### Key top-level fields returned per player

| Field | Type | Description |
|---|---|---|
| `id` | int | NHL player ID (maps to `player_id` in our dim) |
| `firstName.default` | str | First name |
| `lastName.default` | str | Last name |
| `sweaterNumber` | int | Jersey number |
| `positionCode` | str | Position (C/L/R/D/G) |
| `shootsCatches` | str | Handedness (L/R) |
| `heightInInches` | int | Height |
| `weightInPounds` | int | Weight |
| `birthDate` | str | `YYYY-MM-DD` |
| `birthCountry` | str | 3-letter country code |
| `headshot` | str | CDN URL for player headshot |

In [ ]:
resp = httpx.get(f"{NHL_BASE}/roster/{SAMPLE_TEAM}/{SAMPLE_SEASON}", timeout=15)
resp.raise_for_status()
roster_data = resp.json()

print(f"Top-level keys: {list(roster_data.keys())}")
print()

for group in ("forwards", "defensemen", "goalies"):
    players = roster_data.get(group, [])
    print(f"{group}: {len(players)} players")
    if players:
        p0 = players[0]
        print(f"  Player keys : {list(p0.keys())}")
        first = p0.get("firstName", {})
        last  = p0.get("lastName", {})
        name  = f"{first.get('default', '?') if isinstance(first, dict) else first} "\
                f"{last.get('default', '?') if isinstance(last, dict) else last}"
        print(f"  Sample name : {name}")
    print()

# Pick a sample player ID for Section 4
forwards = roster_data.get("forwards", [])
if forwards:
    SAMPLE_PLAYER_ID = forwards[0].get("id")
    first = forwards[0].get("firstName", {})
    last  = forwards[0].get("lastName", {})
    fname = first.get("default", "?") if isinstance(first, dict) else first
    lname = last.get("default", "?") if isinstance(last, dict) else last
    print(f"SAMPLE_PLAYER_ID set to: {SAMPLE_PLAYER_ID} ({fname} {lname})")

## Section 4 — API Exploration: `/v1/player/{player_id}/landing`

Per-player enrichment endpoint. Authoritative for fields that the roster endpoint
cannot supply as explicit columns:

| Field | Why roster endpoint can't provide it |
|---|---|
| `currentTeamId` | Roster gives implicit context (which team you fetched), not an explicit FK |
| `isActive` | Roster only returns active players — can't detect when a player goes inactive |
| `draftDetails` | Not present in roster response |

**Use case:** One landing call per unknown `player_id` during boxscore ingestion.

In [ ]:
if SAMPLE_PLAYER_ID is None:
    raise RuntimeError("SAMPLE_PLAYER_ID not set — run Section 3 first")

resp = httpx.get(f"{NHL_BASE}/player/{SAMPLE_PLAYER_ID}/landing", timeout=15)
resp.raise_for_status()
landing_data = resp.json()

print("Top-level keys in /player/{id}/landing response:")
print(list(landing_data.keys()))
print()

print("Key player dim fields available from landing:")
identity_keys = [
    "playerId", "firstName", "lastName", "sweaterNumber", "position",
    "shootsCatches", "heightInInches", "weightInPounds", "birthDate",
    "birthCountry", "headshot", "currentTeamId", "isActive", "draftDetails",
]
for key in identity_keys:
    val = landing_data.get(key)
    display_val = str(val)[:80] if val is not None else "NULL"
    present = "✓" if val is not None else "✗"
    print(f"  {present} {key:20s}: {display_val}")

## Section 5 — API Exploration: Boxscore Player Identity

The `/v1/gamecenter/{game_id}/boxscore` endpoint is the source for in-game stats.
Per the field inventory from Issue #160, player name fields were **100% null** in
`playerByGameStats`.

This section confirms that finding and documents which player identity fields
the boxscore endpoint does and does not provide.

In [ ]:
resp = httpx.get(f"{NHL_BASE}/gamecenter/{SAMPLE_GAME_ID}/boxscore", timeout=15)
resp.raise_for_status()
boxscore_data = resp.json()

pbgs = boxscore_data.get("playerByGameStats", {})
away_forwards = pbgs.get("awayTeam", {}).get("forwards", [])

print(f"Game : {boxscore_data.get('id')}")
print(f"State: {boxscore_data.get('gameState')}")
print(f"Away forwards in playerByGameStats: {len(away_forwards)}")
print()

if away_forwards:
    p = away_forwards[0]
    print(f"Sample player keys: {list(p.keys())}")
    print()
    print("Player identity fields in boxscore:")
    for key in ["playerId", "sweaterNumber", "name", "position"]:
        val = p.get(key)
        print(f"  {key}: {val}")
    print()
    name_val = p.get("name")
    if isinstance(name_val, dict):
        fn = name_val.get("firstName", {})
        ln = name_val.get("lastName", {})
        fn_val = fn.get("default") if isinstance(fn, dict) else fn
        ln_val = ln.get("default") if isinstance(ln, dict) else ln
        print(f"  name.firstName.default: {fn_val}")
        print(f"  name.lastName.default : {ln_val}")
    else:
        print(f"  name field type: {type(name_val)} — value: {name_val}")
        print("  ← NULL confirms Issue #160 finding: boxscore is not a viable player identity source")

## Section 6 — Batch Fetch: All 32 Teams

Fetches rosters for all 32 NHL teams in the current season via
`/v1/roster/{team}/{season}`. This simulates the full backfill operation
and measures real-world coverage.

**Expected outcome:** ~700–900 unique players across forwards, defensemen, and goalies.

Rate-limited at 50 ms between calls; total runtime ~2 s for the 32 API calls.

In [ ]:
all_players_raw = []
failed_teams = []

for team in NHL_TEAMS:
    try:
        r = httpx.get(f"{NHL_BASE}/roster/{team}/{SAMPLE_SEASON}", timeout=15)
        if r.status_code != 200:
            failed_teams.append((team, f"HTTP {r.status_code}"))
            continue
        data = r.json()
        for group, is_goalie in [("forwards", False), ("defensemen", False), ("goalies", True)]:
            for p in data.get(group, []):
                all_players_raw.append({
                    "source_team"   : team,
                    "position_group": group,
                    "is_goalie"     : is_goalie,
                    **p,
                })
    except Exception as exc:
        failed_teams.append((team, str(exc)))
    time.sleep(0.05)  # polite rate-limiting — 50 ms between requests

print(f"Players collected : {len(all_players_raw)}")
print(f"Failed teams      : {len(failed_teams)}")
if failed_teams:
    for t, reason in failed_teams:
        print(f"  {t}: {reason}")

## Section 7 — Field Inventory

Builds field inventory tables for each source endpoint:
column name, inferred dtype, null rate, unique count, and an example value.

**Three inventories:**
1. Roster endpoint — from the 32-team batch fetch
2. Landing endpoint — from the single sample player call
3. Boxscore player identity — from `playerByGameStats` in the sample game

In [ ]:
def field_inventory(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Build a field inventory table: column, dtype, null_rate, example value."""
    rows = []
    n = len(df)
    for col in df.columns:
        null_rate = df[col].isna().mean() if n > 0 else 1.0
        non_null = df[col].dropna()
        inferred_type = type(non_null.iloc[0]).__name__ if len(non_null) > 0 else "unknown"
        example = non_null.iloc[0] if len(non_null) > 0 else None
        unique_count = df[col].nunique(dropna=True)
        rows.append({
            "column"       : col,
            "dtype"        : str(df[col].dtype),
            "inferred_type": inferred_type,
            "null_rate"    : f"{null_rate:.1%}",
            "unique_count" : unique_count,
            "example"      : str(example)[:60] if example is not None else "NULL",
        })
    inv = pd.DataFrame(rows)
    print(f"\n=== Field Inventory: {label} ===")
    display(inv)
    return inv

In [ ]:
# Flatten all_players_raw into a tidy roster DataFrame
def _get_default(val):
    """Extract .default from a localised NHL name dict, or return the value as-is."""
    return val.get("default") if isinstance(val, dict) else val


def flatten_roster_player(p: dict) -> dict:
    """Extract player dim candidate fields from a roster player object."""
    return {
        "player_id"       : p.get("id"),
        "first_name"      : _get_default(p.get("firstName")),
        "last_name"       : _get_default(p.get("lastName")),
        "sweater_number"  : p.get("sweaterNumber"),
        "position"        : p.get("positionCode"),
        "shoots_catches"  : p.get("shootsCatches"),
        "height_in_inches": p.get("heightInInches"),
        "weight_in_pounds": p.get("weightInPounds"),
        "birth_date"      : p.get("birthDate"),
        "birth_city"      : _get_default(p.get("birthCity")),
        "birth_country"   : p.get("birthCountry"),
        "headshot_url"    : p.get("headshot"),
        "source_team"     : p.get("source_team"),
        "position_group"  : p.get("position_group"),
    }


df_roster = pd.DataFrame([flatten_roster_player(p) for p in all_players_raw])

print(f"df_roster : {df_roster.shape[0]} rows × {df_roster.shape[1]} cols")
print(f"Unique player IDs: {df_roster['player_id'].nunique()}")
print(f"Position breakdown: {df_roster['position'].value_counts().to_dict()}")

inv_roster = field_inventory(df_roster, "Roster endpoint (/v1/roster/{team}/{season})")

In [ ]:
# Flatten the single landing response into a one-row DataFrame
def flatten_landing(data: dict) -> dict:
    """Extract player dim fields from /v1/player/{id}/landing."""
    draft = data.get("draftDetails")
    return {
        "player_id"       : data.get("playerId"),
        "first_name"      : _get_default(data.get("firstName")),
        "last_name"       : _get_default(data.get("lastName")),
        "sweater_number"  : data.get("sweaterNumber"),
        "position"        : data.get("position"),
        "shoots_catches"  : data.get("shootsCatches"),
        "height_in_inches": data.get("heightInInches"),
        "weight_in_pounds": data.get("weightInPounds"),
        "birth_date"      : data.get("birthDate"),
        "birth_country"   : data.get("birthCountry"),
        "headshot_url"    : data.get("headshot"),
        "current_team_id" : data.get("currentTeamId"),
        "is_active"       : data.get("isActive"),
        "draft_details"   : str(draft) if draft else None,
    }


df_landing = pd.DataFrame([flatten_landing(landing_data)])

print(f"df_landing : {df_landing.shape[0]} row (sample player)")
inv_landing = field_inventory(df_landing, "Landing endpoint (/v1/player/{id}/landing)")

In [ ]:
# Extract player identity fields from playerByGameStats in the sample boxscore
boxscore_identity_rows = []
pbgs = boxscore_data.get("playerByGameStats", {})
for side_key in ("awayTeam", "homeTeam"):
    side_data = pbgs.get(side_key, {})
    for group in ("forwards", "defense", "goalies"):
        for p in side_data.get(group, []):
            name_field = p.get("name")
            first_name = None
            last_name  = None
            if isinstance(name_field, dict):
                fn = name_field.get("firstName", {})
                ln = name_field.get("lastName", {})
                first_name = fn.get("default") if isinstance(fn, dict) else fn
                last_name  = ln.get("default") if isinstance(ln, dict) else ln
            boxscore_identity_rows.append({
                "player_id"     : p.get("playerId"),
                "sweater_number": p.get("sweaterNumber"),
                "first_name"    : first_name,
                "last_name"     : last_name,
                "position"      : p.get("position"),
            })

df_boxscore_id = pd.DataFrame(boxscore_identity_rows)
print(f"df_boxscore_id : {df_boxscore_id.shape[0]} rows")

if not df_boxscore_id.empty:
    fn_null = df_boxscore_id["first_name"].isna().mean()
    ln_null = df_boxscore_id["last_name"].isna().mean()
    print(f"  first_name null rate : {fn_null:.1%}  (Issue #160 finding: expect 100%)")
    print(f"  last_name  null rate : {ln_null:.1%}")

inv_boxscore = field_inventory(df_boxscore_id, "Boxscore player identity (/v1/gamecenter/{id}/boxscore)")

## Section 8 — Side-by-Side Comparison

Each row is a candidate `player` dim field. Columns show whether the field is
present and reliable in each source endpoint.

| Field | Roster `/v1/roster/` | Landing `/v1/player/` | Boxscore `playerByGameStats` | Best source |
|---|---|---|---|---|
| `player_id` | ✓ `id` | ✓ `playerId` | ✓ `playerId` | Either |
| `first_name` | ✓ `firstName.default` | ✓ `firstName.default` | ✗ 100% null | Roster |
| `last_name` | ✓ `lastName.default` | ✓ `lastName.default` | ✗ 100% null | Roster |
| `sweater_number` | ✓ `sweaterNumber` | ✓ `sweaterNumber` | ✓ `sweaterNumber` | Roster |
| `position` | ✓ `positionCode` | ✓ `position` | ✓ `position` | Roster |
| `shoots_catches` | ✓ `shootsCatches` | ✓ `shootsCatches` | ✗ not present | Roster |
| `height_in_inches` | ✓ `heightInInches` | ✓ `heightInInches` | ✗ not present | Roster |
| `weight_in_pounds` | ✓ `weightInPounds` | ✓ `weightInPounds` | ✗ not present | Roster |
| `birth_date` | ✓ `birthDate` | ✓ `birthDate` | ✗ not present | Roster |
| `birth_country` | ✓ `birthCountry` | ✓ `birthCountry` | ✗ not present | Roster |
| `headshot_url` | ✓ `headshot` | ✓ `headshot` | ✗ not present | Roster |
| `current_team_id` | ✗ implicit only | ✓ `currentTeamId` | ✗ not present | **Landing only** |
| `is_active` | ✗ not present | ✓ `isActive` | ✗ not present | **Landing only** |
| `draft_details` | ✗ not present | ✓ `draftDetails` | ✗ not present | **Landing only** |

**Summary:** The roster endpoint wins for bulk biographical data (32 calls ≈ entire NHL).
The landing endpoint is the only source for `current_team_id`, `is_active`, and draft details.
The boxscore is **not a viable player identity source**.

In [ ]:
# Build a programmatic comparison DataFrame to supplement the markdown table above
fields = [
    ("player_id",         True,  True,  True),
    ("first_name",        True,  True,  False),
    ("last_name",         True,  True,  False),
    ("sweater_number",    True,  True,  True),
    ("position",          True,  True,  True),
    ("shoots_catches",    True,  True,  False),
    ("height_in_inches",  True,  True,  False),
    ("weight_in_pounds",  True,  True,  False),
    ("birth_date",        True,  True,  False),
    ("birth_country",     True,  True,  False),
    ("headshot_url",      True,  True,  False),
    ("current_team_id",   False, True,  False),
    ("is_active",         False, True,  False),
    ("draft_details",     False, True,  False),
]

df_comparison = pd.DataFrame(
    fields,
    columns=["field", "roster", "landing", "boxscore"],
)
df_comparison["best_source"] = df_comparison.apply(
    lambda r: "landing" if r["landing"] and not r["roster"] else
              ("roster" if r["roster"] else "none"),
    axis=1,
)

print("Side-by-side comparison: which endpoint provides each player dim field")
display(df_comparison)

roster_only_fields  = df_comparison[df_comparison["roster"] & ~df_comparison["landing"]]["field"].tolist()
landing_only_fields = df_comparison[df_comparison["landing"] & ~df_comparison["roster"]]["field"].tolist()
print(f"\nRoster-only fields  : {roster_only_fields}")
print(f"Landing-only fields : {landing_only_fields}")

## Section 9 — Proposed Player Dimension DDL

Based on the side-by-side comparison, the best available source for each column
is documented in the inline comments.

```sql
CREATE TABLE player (
    -- Core identity — best source: /v1/roster/{team}/{season}
    player_id         INTEGER  PRIMARY KEY,   -- API: id (roster) / playerId (landing)
    first_name        TEXT,                   -- API: firstName.default
    last_name         TEXT,                   -- API: lastName.default
    sweater_number    INTEGER,                -- API: sweaterNumber (latest observed)
    position          TEXT,                   -- API: positionCode (roster) / position (landing)
    shoots_catches    TEXT,                   -- API: shootsCatches (L/R)
    height_in_inches  INTEGER,               -- API: heightInInches
    weight_in_pounds  INTEGER,               -- API: weightInPounds
    birth_date        TEXT,                   -- API: birthDate (YYYY-MM-DD)
    birth_country     TEXT,                   -- API: birthCountry (3-letter code)
    headshot_url      TEXT,                   -- API: headshot (CDN URL)

    -- Enrichment — best source: /v1/player/{id}/landing
    current_team_id   INTEGER,               -- API: currentTeamId (authoritative)
    is_active         INTEGER,               -- API: isActive; 1=active, 0=inactive

    -- Audit
    updated_at        TEXT                    -- ISO 8601 timestamp of last upsert
);
```

### Design decisions

| Decision | Rationale |
|---|---|
| `is_active` as INTEGER | SQLite has no BOOLEAN; 1/0 convention used throughout the project |
| `sweater_number` overwritten on upsert | Numbers change between seasons — always store the latest observed value |
| `current_team_id` not set during roster backfill | Roster gives implicit context only; use landing for the authoritative FK |
| `draft_details` excluded | Complex nested object; not required for MVP dashboard |
| `birth_city` excluded | Low signal for dashboard use case; can be added later if needed |

## Section 10 — Backfill & Ongoing Upsert Workflow

### (a) Initial Backfill — 32 roster calls to populate the full player dim

```
for each team in NHL_TEAMS (32 teams):
    1. GET /v1/roster-season/{team}
       → confirms which seasons have roster data
    2. GET /v1/roster/{team}/20252026
       → returns forwards + defensemen + goalies with full biographical data
    3. For each player in the response:
       → upsert into player dim by player_id
       → set all biographical columns
       → current_team_id and is_active left NULL (set in enrichment pass)
```

**Coverage:** 32 calls × ~25 players/team ≈ 800 records covering all active NHL players.  
**Rate limit:** 50 ms between calls → ~2 s total for the backfill.

### (b) Ongoing Upsert — triggered by unknown `player_id` during boxscore ingestion

```
During boxscore ingestion for game G:
    for each player_id P in playerByGameStats:
        if P NOT IN player dim:
            1. GET /v1/player/{P}/landing
               → upsert player dim with all fields including current_team_id and is_active
        else:
            pass  # player already known — no action required
```

**Rate:** New players appear infrequently (trades, call-ups, debuts). Expect < 5
landing calls per game day on average.

### Upsert SQL pattern

```sql
INSERT INTO player (player_id, first_name, last_name, sweater_number,
                    position, shoots_catches, height_in_inches,
                    weight_in_pounds, birth_date, birth_country,
                    headshot_url, current_team_id, is_active, updated_at)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
ON CONFLICT(player_id) DO UPDATE SET
    first_name        = excluded.first_name,
    last_name         = excluded.last_name,
    sweater_number    = excluded.sweater_number,
    position          = excluded.position,
    shoots_catches    = excluded.shoots_catches,
    height_in_inches  = excluded.height_in_inches,
    weight_in_pounds  = excluded.weight_in_pounds,
    birth_date        = excluded.birth_date,
    birth_country     = excluded.birth_country,
    headshot_url      = excluded.headshot_url,
    current_team_id   = COALESCE(excluded.current_team_id, player.current_team_id),
    is_active         = COALESCE(excluded.is_active, player.is_active),
    updated_at        = excluded.updated_at;
```

**Note on COALESCE:** The roster backfill does not supply `current_team_id` or
`is_active`, so they arrive as NULL. `COALESCE` preserves the existing value
from a prior landing call rather than overwriting it with NULL.

## Section 11 — Gap Analysis

### Fields only available from `/v1/player/{id}/landing`

| Field | Why it matters |
|---|---|
| `currentTeamId` | Authoritative team FK; roster endpoint gives only implicit context |
| `isActive` | Critical for filtering retired/inactive players from active-only queries |
| `draftDetails` | Draft round, year, and pick; not available in roster or boxscore |

### Fields the roster endpoint covers (that landing also covers)

Both provide: `firstName`, `lastName`, `sweaterNumber`, `positionCode/position`,
`shootsCatches`, `heightInInches`, `weightInPounds`, `birthDate`, `birthCountry`,
`headshot`.

**Recommendation:** Use roster for bulk backfill (32 calls vs. ~800 per active
player), and landing only for the three enrichment-only fields.

### Boxscore — not a viable player identity source

| Reason | Detail |
|---|---|
| Names 100% null | Confirmed by Issue #160 field inventory |
| No biographical data | Height, weight, birth info — all absent |
| `playerId` present | Useful only as a foreign key lookup trigger |
| `sweaterNumber` present | Useful as a secondary identity signal |

### Conclusion

The optimal player dim population strategy in three steps:

1. **Bulk backfill (one-time):** 32 × `/v1/roster/{team}/20252026` → ~800 records with full biography
2. **Enrichment pass:** per-player landing call for `currentTeamId` + `isActive`
3. **Maintenance:** during boxscore ingestion, new `player_id` values trigger an on-demand landing call

This minimises API calls while ensuring freshness of team affiliation and active status.

In [ ]:
# Quantify the gap between roster and landing coverage
roster_fields = {
    "player_id", "first_name", "last_name", "sweater_number",
    "position", "shoots_catches", "height_in_inches", "weight_in_pounds",
    "birth_date", "birth_country", "headshot_url",
}
landing_fields = {
    "player_id", "first_name", "last_name", "sweater_number",
    "position", "shoots_catches", "height_in_inches", "weight_in_pounds",
    "birth_date", "birth_country", "headshot_url",
    "current_team_id", "is_active", "draft_details",
}
boxscore_fields = {"player_id", "sweater_number", "position"}  # names are NULL

landing_only_fields  = landing_fields - roster_fields
roster_only_fields   = roster_fields - landing_fields
both_fields          = roster_fields & landing_fields
boxscore_gap_fields  = (roster_fields | landing_fields) - boxscore_fields

print("=== Field Coverage Gap Analysis ===")
print(f"\nFields in BOTH roster + landing ({len(both_fields)}):")
for f in sorted(both_fields):
    print(f"  {f}")

print(f"\nFields in landing ONLY — not in roster ({len(landing_only_fields)}):")
for f in sorted(landing_only_fields):
    print(f"  {f}  ← requires /v1/player/{{id}}/landing call")

print(f"\nFields in roster ONLY — not in landing ({len(roster_only_fields) or 0}):")
if roster_only_fields:
    for f in sorted(roster_only_fields):
        print(f"  {f}")
else:
    print("  (none — landing is a superset of roster for player dim fields)")

print(f"\nFields MISSING from boxscore player identity ({len(boxscore_gap_fields)}):")
for f in sorted(boxscore_gap_fields):
    print(f"  {f}")

print("\n=== Summary ===")
print("Use roster for bulk backfill; landing for current_team_id + is_active.")
print("Boxscore is not a viable player identity source (names 100% null, no bio fields).")